In [1]:
import sys
import os

sys.path.append("..")
sys.path.append("../..")
sys.path.append("../../..")

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import sklearn

from experiment.utils.utils import logit

from typing import Literal

In [3]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [4]:
import hydra
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra


GlobalHydra.instance().clear()
initialize(version_base=None, config_path="../../config")

CONFIG = compose(config_name="config")

# Environment Sampling

In [5]:
NUM_AUCTIONS = 600

In [6]:
def price2bin(price):
    """
    price = 1.2 ** bin
    """
    if price <= 0:
        return 0
    return np.round(np.log(price) / np.log(1.2))

def bin2price(bin_):
    return np.power(1.2, bin_)

In [7]:
from datasets import load_dataset, load_from_disk
if not os.path.exists("../../data/BAT/vcg_campaigns") or not os.path.exists("../../data/BAT/vcg_stats"):
    vcg_campaigns = load_dataset("AvitoTech/BAT", "vcg_campaigns")
    vcg_stats = load_dataset("AvitoTech/BAT", "vcg_stats")

    vcg_campaigns.save_to_disk("../../data/BAT/vcg_campaigns")
    vcg_stats.save_to_disk("../../data/BAT/vcg_stats")

/Users/iazhigalsky/work/denoise-bid-uai26/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
VCG_CAMPAIGNS = load_from_disk("../../data/BAT/vcg_campaigns")
VCG_STATS = load_from_disk("../../data/BAT/vcg_stats")

In [14]:
VCG_STATS_SUBSAMPLE_DF = pd.DataFrame(VCG_STATS["train"][:10_000_000])
VCG_STATS_SUBSAMPLE_DF = VCG_STATS_SUBSAMPLE_DF[VCG_STATS_SUBSAMPLE_DF["CTRPredicts"] >= 1e-9].reset_index()
VCG_CAMPAIGNS_DF = pd.DataFrame(VCG_CAMPAIGNS["train"])

unique_campaign_ids = VCG_STATS_SUBSAMPLE_DF["campaign_id"].unique()

VCG_CAMPAIGNS_SUBSAMPLE_DF = VCG_CAMPAIGNS_DF[
    VCG_CAMPAIGNS_DF["campaign_id"].isin(unique_campaign_ids)
].reset_index()

In [15]:
mask = VCG_STATS_SUBSAMPLE_DF.groupby("campaign_id")["AuctionCount"].transform("count") >= NUM_AUCTIONS
VCG_STATS_SUBSAMPLE_DF_FILTERED = VCG_STATS_SUBSAMPLE_DF[mask].copy().reset_index(drop=True)
CAMPAIGN_IDS = VCG_STATS_SUBSAMPLE_DF_FILTERED["campaign_id"].unique()

In [16]:
CAMPAIGN_IDS = CAMPAIGN_IDS[:100]

In [17]:
def sample_campaign(
    seed,
    campaign_id,
    sigma_ctr = 0.0,
    sigma_cvr = 0.0,
    vcg_stats_df = VCG_STATS_SUBSAMPLE_DF_FILTERED,
):
    rng = np.random.default_rng(seed)
    campaign = vcg_stats_df[vcg_stats_df["campaign_id"] == campaign_id]

    weights = np.array(campaign["AuctionCount"], dtype=np.float64)
    weights /= weights.sum()

    idxs = rng.choice(len(campaign), size=len(campaign), p=weights, replace=True)
    logs = vcg_stats_df.iloc[idxs]

    place_count = logs["AuctionCount"]
    ctr_clean = np.array(logs["CTRPredicts"] / place_count)
    cvr_clean = np.array(logs["CRPredicts"] / place_count)
    wp = np.array(bin2price(logs["contact_price_bin"]) / place_count)

    ctr_clean_logit = logit(ctr_clean)
    cvr_clean_logit = logit(cvr_clean)

    ctr_noised_logit = ctr_clean_logit + rng.normal(0, sigma_ctr, len(campaign))
    cvr_noised_logit = cvr_clean_logit + rng.normal(0, sigma_cvr, len(campaign))

    return ctr_clean_logit, ctr_noised_logit, cvr_clean_logit, cvr_noised_logit, wp

In [18]:
from experiment.utils.utils import sigmoid

In [19]:
from time import time

In [20]:
SEED = 42

## latency tests

In [21]:
from experiment.non_robust_bid.lp import solve_dual as non_robust_dual
from experiment.non_robust_bid.bid import bids as non_robust_bids

def time_test_non_robust_bid(camp_ids=CAMPAIGN_IDS, sigma_ctr=1., sigma_cvr=1., budget_share=0.2, cpc_share=0.2):
    latencies = np.zeros(len(camp_ids))
    for i, campaign_id in tqdm(enumerate(camp_ids), total=len(camp_ids)):

        ctr_clean_logit, ctr_noised_logit, cvr_clean_logit, cvr_noised_logit, wp = sample_campaign(
            seed=SEED,
            campaign_id=campaign_id,
            sigma_ctr=sigma_ctr,
            sigma_cvr=sigma_cvr,
        )

        budget = budget_share * np.sum(wp)
        target_cpc = cpc_share * np.sum(wp) / np.sum(sigmoid(ctr_noised_logit) * wp)

        ctr = sigmoid(ctr_noised_logit)
        cvr = sigmoid(cvr_noised_logit)

        p, q = non_robust_dual(
            config=CONFIG.algorithms.non_robust_bid,
            ctr=ctr,
            cvr=cvr,
            wp=wp,
            budget=budget,
            target_cpc=target_cpc,
        )

        start = time()
        _ = non_robust_bids(
            ctr[:1], cvr[:1], p, q, target_cpc
        )
        latency = time() - start
        latencies[i] = latency

    return latencies

In [22]:
from experiment.robust_bid.lp import solve_dual as robust_dual
from experiment.robust_bid.bid import bids as robust_bids

def time_test_robust_bid(camp_ids=CAMPAIGN_IDS, sigma_ctr=1., sigma_cvr=1., budget_share=0.2, cpc_share=0.2):
    latencies = np.zeros(len(camp_ids))
    for j, campaign_id in tqdm(enumerate(camp_ids), total=len(camp_ids)):

        ctr_clean_logit, ctr_noised_logit, _, cvr_noised_logit, wp = sample_campaign(
            seed=SEED,
            campaign_id=campaign_id,
            sigma_ctr=sigma_ctr,
            sigma_cvr=sigma_cvr,
        )

        budget = budget_share * np.sum(wp)
        target_cpc = cpc_share * np.sum(wp) / np.sum(sigmoid(ctr_noised_logit) * wp)

        ctr_clean = sigmoid(ctr_clean_logit)
        ctr_noised = sigmoid(ctr_noised_logit)
        cvr_noised = sigmoid(cvr_noised_logit)

        epsilon = np.sqrt(np.sum((ctr_clean - ctr_noised) ** 2))

        p, q, delta, u = robust_dual(
            config=CONFIG.algorithms.robust_bid,
            ctr=ctr_noised,
            cvr=cvr_noised,
            wp=wp,
            budget=budget,
            target_cpc=target_cpc,
            epsilon=epsilon,
        )

        start = time()
        for i in range(len(ctr_noised)):
            _ = robust_bids(
                ctr_noised[i:i+1], cvr_noised[i:i+1], p, q, delta[i:i+1], u[i:i+1], target_cpc, epsilon
            )
        latency = (time() - start) / len(ctr_noised)
        latencies[j] = latency

    return latencies

In [23]:
from experiment.denoise_bid.ctr_only.gmm import fit_gmm as fit_gmm_ctr_only
from experiment.denoise_bid.ctr_only.expectation import ctr_expectation as ctr_expectation_ctr_only
from experiment.denoise_bid.ctr_only.lp import solve_dual as denoise_ctr_only_dual
from experiment.denoise_bid.ctr_only.bid import bids as denoise_ctr_only_bids

def time_test_denoise_bid_ctr_only(camp_ids=CAMPAIGN_IDS, sigma_ctr=1., sigma_cvr=1., budget_share=0.2, cpc_share=0.2, n_components=1):
    latencies = np.zeros(len(camp_ids))
    for j, campaign_id in tqdm(enumerate(camp_ids), total=len(camp_ids)):

        _, ctr_noised_logit, _, cvr_noised_logit, wp = sample_campaign(
            seed=SEED,
            campaign_id=campaign_id,
            sigma_ctr=sigma_ctr,
            sigma_cvr=sigma_cvr,
        )

        budget = budget_share * np.sum(wp)
        target_cpc = cpc_share * np.sum(wp) / np.sum(sigmoid(ctr_noised_logit) * wp)

        indexes = np.random.choice(
            len(ctr_noised_logit),
            size=200 * n_components,
            replace=False,
        )

        ctr_sigmas = np.ones_like(ctr_noised_logit) * sigma_ctr

        ctr_gmm_logit = ctr_noised_logit[indexes]
        ctr_gmm_sigma = ctr_sigmas[indexes]
        weights, means, sigmas = fit_gmm_ctr_only(
            CONFIG.algorithms.denoise_bid.ctr_only,
            ctr_gmm_logit,
            ctr_gmm_sigma,
            n_components,
        )

        start = time()
        for i in range(len(ctr_noised_logit)):
            ctr = ctr_expectation_ctr_only(ctr_noised_logit[i:i+1], ctr_sigmas[i:i+1], weights, means, sigmas)
        expectation_latency = time() - start

        ctr = ctr_expectation_ctr_only(ctr_noised_logit, ctr_sigmas, weights, means, sigmas)
        cvr = sigmoid(cvr_noised_logit)
        p, q = denoise_ctr_only_dual(CONFIG.algorithms.denoise_bid.ctr_only, ctr, cvr, wp, budget, target_cpc)

        start = time()
        for i in range(len(ctr_noised_logit)):
            _ = denoise_ctr_only_bids(ctr[i:i+1], cvr[i:i+1], p, q, target_cpc)
        bid_latency = time() - start
        latencies[j] = (bid_latency + expectation_latency) / len(ctr_noised_logit)

    return latencies

In [24]:
from experiment.denoise_bid.joint.gmm import fit_gmm as fit_gmm_joint
from experiment.denoise_bid.joint.expectation import ctr_expectation as ctr_expectation_joint
from experiment.denoise_bid.joint.expectation import value_expectation as value_expectation_joint
from experiment.denoise_bid.joint.lp import solve_dual as denoise_joint_dual
from experiment.denoise_bid.joint.bid import bids as denoise_joint_bids

def time_test_denoise_bid_joint(camp_ids=CAMPAIGN_IDS, sigma_ctr=1., sigma_cvr=1., budget_share=0.2, cpc_share=0.2, n_components=1, quad_points=3):
    latencies = np.zeros(len(camp_ids))
    for j, campaign_id in tqdm(enumerate(camp_ids), total=len(camp_ids)):

        _, ctr_noised_logit, _, cvr_noised_logit, wp = sample_campaign(
            seed=SEED,
            campaign_id=campaign_id,
            sigma_ctr=sigma_ctr,
            sigma_cvr=sigma_cvr,
        )

        budget = budget_share * np.sum(wp)
        target_cpc = cpc_share * np.sum(wp) / np.sum(sigmoid(ctr_noised_logit) * wp)

        ctr_sigmas = np.ones_like(ctr_noised_logit) * sigma_ctr
        cvr_sigmas = np.ones_like(cvr_noised_logit) * sigma_cvr

        indexes = np.random.choice(
            len(ctr_noised_logit),
            size=200 * n_components,
            replace=False
        )
        ctr_gmm_logit = ctr_noised_logit[indexes]
        cvr_gmm_logit = cvr_noised_logit[indexes]
        ctr_gmm_sigma = ctr_sigmas[indexes]
        cvr_gmm_sigma = cvr_sigmas[indexes]
        weights, means, sigmas = fit_gmm_joint(
            CONFIG.algorithms.denoise_bid.joint,
            ctr_gmm_logit,
            cvr_gmm_logit,
            ctr_gmm_sigma,
            cvr_gmm_sigma,
            n_components,
        )

        herm_gaus_points = np.polynomial.hermite.hermgauss(CONFIG.algorithms.denoise_bid.joint.gh_approximation.n_quad_points)

        start = time()
        for i in range(len(ctr_noised_logit)):
            ctr = ctr_expectation_joint(
                ctr_noised_logit[i:i+1],
                cvr_noised_logit[i:i+1],
                ctr_sigmas[i:i+1],
                cvr_sigmas[i:i+1],
                weights,
                means,
                sigmas,
            )
            value = value_expectation_joint(
                ctr_noised_logit[i:i+1],
                cvr_noised_logit[i:i+1],
                ctr_sigmas[i:i+1],
                cvr_sigmas[i:i+1],
                weights,
                means,
                sigmas,
                CONFIG.algorithms.denoise_bid.joint.gh_approximation.n_quad_points,
                herm_gaus_points,
            )
        expectation_latency = time() - start

        ctr = ctr_expectation_joint(
            ctr_noised_logit,
            cvr_noised_logit,
            ctr_sigmas,
            cvr_sigmas,
            weights,
            means,
            sigmas,
        )
        value = value_expectation_joint(
            ctr_noised_logit,
            cvr_noised_logit,
            ctr_sigmas,
            cvr_sigmas,
            weights,
            means,
            sigmas,
            CONFIG.algorithms.denoise_bid.joint.gh_approximation.n_quad_points,
            herm_gaus_points,
        )
        p, q = denoise_joint_dual(
            CONFIG.algorithms.denoise_bid.joint,
            ctr,
            value,
            wp,
            budget,
            target_cpc,
        )

        start = time()
        for i in range(len(ctr_noised_logit)):
            _ = denoise_joint_bids(ctr[i:i+1], value[i:i+1], p, q, target_cpc)
        bids_latency = time() - start
        latencies[j] = (bids_latency + expectation_latency) / len(ctr_noised_logit)

    return latencies

## TEST

In [25]:
non_robust_latencies = time_test_non_robust_bid()

print(
    f"Runtime (mean ± SEM): {non_robust_latencies.mean() * 10**6:.6f} ± {non_robust_latencies.std()/np.sqrt(len(non_robust_latencies)) * 10**6:.6f} μs, "
    f"median: {np.median(non_robust_latencies) * 10**6:.6f} μs, "
    f"p99: {np.percentile(non_robust_latencies, 99) * 10**6:.6f} μs, "
    f"std: {non_robust_latencies.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [00:00<00:00, 137.88it/s]

Runtime (mean ± SEM): 3.228188 ± 0.137186 μs, median: 3.099442 μs, p99: 6.995201 μs, std: 1.371858 μs


In [26]:
robust_latencies = time_test_robust_bid()

print(
    f"Runtime (mean ± SEM): {robust_latencies.mean() * 10**6:.6f} ± {robust_latencies.std()/np.sqrt(len(robust_latencies)) * 10**6:.6f} μs, "
    f"median: {np.median(robust_latencies) * 10**6:.6f} μs, "
    f"p99: {np.percentile(robust_latencies, 99) * 10**6:.6f} μs, "
    f"std: {robust_latencies.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [00:02<00:00, 47.99it/s]

Runtime (mean ± SEM): 3.106746 ± 0.036206 μs, median: 3.035432 μs, p99: 3.572569 μs, std: 0.362064 μs


In [27]:
denoise_latencies_1_comp = time_test_denoise_bid_ctr_only(n_components=1)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_1_comp.mean() * 10**6:.6f} ± {denoise_latencies_1_comp.std()/np.sqrt(len(denoise_latencies_1_comp)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_1_comp) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_1_comp, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_1_comp.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [00:08<00:00, 12.15it/s]

Runtime (mean ± SEM): 13.810714 ± 0.133008 μs, median: 13.563455 μs, p99: 17.505678 μs, std: 1.330082 μs


In [28]:
denoise_latencies_2_comp = time_test_denoise_bid_ctr_only(n_components=2)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_2_comp.mean() * 10**6:.6f} ± {denoise_latencies_2_comp.std()/np.sqrt(len(denoise_latencies_2_comp)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_2_comp) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_2_comp, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_2_comp.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [03:50<00:00,  2.30s/it]

Runtime (mean ± SEM): 13.802306 ± 0.011509 μs, median: 13.788973 μs, p99: 14.148695 μs, std: 0.115090 μs


In [29]:
denoise_latencies_3_comp = time_test_denoise_bid_ctr_only(n_components=3)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_3_comp.mean() * 10**6:.6f} ± {denoise_latencies_3_comp.std()/np.sqrt(len(denoise_latencies_3_comp)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_3_comp) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_3_comp, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_3_comp.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [08:45<00:00,  5.26s/it]

Runtime (mean ± SEM): 13.812499 ± 0.016665 μs, median: 13.792816 μs, p99: 14.258836 μs, std: 0.166645 μs


In [30]:
denoise_latencies_1_comp_3_quad_points = time_test_denoise_bid_joint(n_components=1, quad_points=3)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_1_comp_3_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_1_comp_3_quad_points.std()/np.sqrt(len(denoise_latencies_1_comp_3_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_1_comp_3_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_1_comp_3_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_1_comp_3_quad_points.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [00:40<00:00,  2.45it/s]

Runtime (mean ± SEM): 156.596526 ± 1.029925 μs, median: 149.890768 μs, p99: 179.906424 μs, std: 10.299249 μs


In [31]:
denoise_latencies_2_comp_3_quad_points = time_test_denoise_bid_joint(n_components=2, quad_points=3)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_2_comp_3_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_2_comp_3_quad_points.std()/np.sqrt(len(denoise_latencies_1_comp_3_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_2_comp_3_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_2_comp_3_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_2_comp_3_quad_points.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [05:56<00:00,  3.56s/it]

Runtime (mean ± SEM): 175.497972 ± 0.312910 μs, median: 174.913482 μs, p99: 184.914293 μs, std: 3.129097 μs


In [32]:
denoise_latencies_3_comp_3_quad_points = time_test_denoise_bid_joint(n_components=3, quad_points=3)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_3_comp_3_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_3_comp_3_quad_points.std()/np.sqrt(len(denoise_latencies_1_comp_3_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_3_comp_3_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_3_comp_3_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_3_comp_3_quad_points.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [13:00<00:00,  7.80s/it]

Runtime (mean ± SEM): 198.211699 ± 0.720083 μs, median: 196.463366 μs, p99: 227.741257 μs, std: 7.200829 μs


In [33]:
denoise_latencies_1_comp_5_quad_points = time_test_denoise_bid_joint(n_components=1, quad_points=5)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_1_comp_5_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_1_comp_5_quad_points.std()/np.sqrt(len(denoise_latencies_1_comp_5_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_1_comp_5_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_1_comp_5_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_1_comp_5_quad_points.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [00:40<00:00,  2.48it/s]

Runtime (mean ± SEM): 158.618628 ± 1.454351 μs, median: 149.377427 μs, p99: 211.937977 μs, std: 14.543511 μs


In [34]:
denoise_latencies_2_comp_5_quad_points = time_test_denoise_bid_joint(n_components=2, quad_points=5)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_2_comp_5_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_2_comp_5_quad_points.std()/np.sqrt(len(denoise_latencies_2_comp_5_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_2_comp_5_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_2_comp_5_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_2_comp_5_quad_points.std() * 10**6:.6f} μs"
)

100%|██████████| 100/100 [05:58<00:00,  3.58s/it]

Runtime (mean ± SEM): 175.779461 ± 0.430395 μs, median: 174.805565 μs, p99: 197.107916 μs, std: 4.303953 μs


In [ ]:
denoise_latencies_3_comp_5_quad_points = time_test_denoise_bid_joint(n_components=3, quad_points=5)

print(
    f"Runtime (mean ± SEM): {denoise_latencies_3_comp_5_quad_points.mean() * 10**6:.6f} ± {denoise_latencies_3_comp_5_quad_points.std()/np.sqrt(len(denoise_latencies_3_comp_5_quad_points)) * 10**6:.6f} μs, "
    f"median: {np.median(denoise_latencies_3_comp_5_quad_points) * 10**6:.6f} μs, "
    f"p99: {np.percentile(denoise_latencies_3_comp_5_quad_points, 99) * 10**6:.6f} μs, "
    f"std: {denoise_latencies_3_comp_5_quad_points.std() * 10**6:.6f} μs"
)

  3%|▎         | 3/100 [00:23<12:53,  7.98s/it]